Install Java (Spark needs it)

In [20]:
!apt-get install -y openjdk-17-jdk -q
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
print("✅ Java installed")
!java -version

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libgail-common libgail18 libgtk2.0-0
  libgtk2.0-bin libgtk2.0-common librsvg2-common libxcomposite1 libxt-dev
  libxtst6 libxxf86dga1 openjdk-17-jre session-migration x11-utils
Suggested packages:
  gvfs libxt-doc openjdk-17-demo openjdk-17-source visualvm mesa-utils
The following NEW packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libgail-common libgail18 libgtk2.0-0
  libgtk2.0-bin libgtk2.0-common librsvg2-common libxcomposite1 libxt-dev
  libxtst6 libxxf86dga1 openjdk-17-jdk openjdk-17-jre session-migra

Install PySpark

In [21]:
!pip install pyspark -q
print("✅ PySpark installed")

✅ PySpark installed


Create a SparkSession with UI enabled on a safe port

In [22]:
from pyspark.sql import SparkSession
import os

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

spark = SparkSession.builder \
    .appName("MyFirstSparkApp") \
    .config("spark.ui.port", "4041") \
    .config("spark.ui.enabled", "true") \
    .getOrCreate()

print("✅ Spark started!")
print(f"Spark version: {spark.version}")

✅ Spark started!
Spark version: 4.0.2


In [26]:
print(f"Spark UI URL: {spark.sparkContext.uiWebUrl}")

Spark UI URL: http://dea69ca67dc4:4050


Expose Spark UI via ngrok

In [27]:
from pyngrok import ngrok

ngrok.set_auth_token("")

# Kill old tunnels first
ngrok.kill()

import time
time.sleep(2)

spark_ui_url = ngrok.connect(4050)
print(f"✅ Spark UI is live at: {spark_ui_url}")

✅ Spark UI is live at: NgrokTunnel: "https://smuggling-either-bunkbed.ngrok-free.dev" -> "http://localhost:4050"


Spark job

In [28]:
# Create a simple dataset
data = [
    ("Alice", 25, "Engineer"),
    ("Bob", 30, "Data Scientist"),
    ("Charlie", 35, "Manager"),
    ("Diana", 28, "Engineer"),
    ("Eve", 32, "Data Scientist"),
]

# Create a DataFrame
columns = ["name", "age", "role"]
df = spark.createDataFrame(data, columns)

# Show the data
print("📋 Full Dataset:")
df.show()

# Filter Engineers only
print("👷 Engineers only:")
df.filter(df.role == "Engineer").show()

# Group by role and count
print("📊 Count by Role:")
df.groupBy("role").count().show()

📋 Full Dataset:
+-------+---+--------------+
|   name|age|          role|
+-------+---+--------------+
|  Alice| 25|      Engineer|
|    Bob| 30|Data Scientist|
|Charlie| 35|       Manager|
|  Diana| 28|      Engineer|
|    Eve| 32|Data Scientist|
+-------+---+--------------+

👷 Engineers only:
+-----+---+--------+
| name|age|    role|
+-----+---+--------+
|Alice| 25|Engineer|
|Diana| 28|Engineer|
+-----+---+--------+

📊 Count by Role:
+--------------+-----+
|          role|count|
+--------------+-----+
|Data Scientist|    2|
|      Engineer|    2|
|       Manager|    1|
+--------------+-----+



In [29]:
print(f"🔗 Open your Spark UI at: {spark_ui_url}")
print("In the UI you can see:")
print("  - Jobs tab: all completed/running jobs")
print("  - Stages tab: stages within each job")
print("  - Storage tab: cached data")
print("  - Executors tab: resource usage")

🔗 Open your Spark UI at: NgrokTunnel: "https://smuggling-either-bunkbed.ngrok-free.dev" -> "http://localhost:4050"
In the UI you can see:
  - Jobs tab: all completed/running jobs
  - Stages tab: stages within each job
  - Storage tab: cached data
  - Executors tab: resource usage


In [30]:
# Create a sample CSV file first
import pandas as pd

pdf = pd.DataFrame({
    'product': ['Apple', 'Banana', 'Cherry', 'Apple', 'Banana'],
    'sales': [100, 200, 150, 300, 250],
    'region': ['North', 'South', 'North', 'South', 'North']
})
pdf.to_csv('/root/sales.csv', index=False)

# Read it with Spark
df_sales = spark.read.csv('/root/sales.csv', header=True, inferSchema=True)

print("📋 Sales Data:")
df_sales.show()

print("💰 Total Sales by Product:")
df_sales.groupBy("product").sum("sales").show()

print("🌍 Total Sales by Region:")
df_sales.groupBy("region").sum("sales").show()

📋 Sales Data:
+-------+-----+------+
|product|sales|region|
+-------+-----+------+
|  Apple|  100| North|
| Banana|  200| South|
| Cherry|  150| North|
|  Apple|  300| South|
| Banana|  250| North|
+-------+-----+------+

💰 Total Sales by Product:
+-------+----------+
|product|sum(sales)|
+-------+----------+
| Banana|       450|
| Cherry|       150|
|  Apple|       400|
+-------+----------+

🌍 Total Sales by Region:
+------+----------+
|region|sum(sales)|
+------+----------+
| South|       500|
| North|       500|
+------+----------+

